# Protein Prioritization Ranking

In [ ]:
import jsonfrom pathlib import Pathfrom typing import Dict, List, Optional, Tuple

## ProteinRankingIntegrator

In [ ]:
class ProteinRankingIntegrator:    """Integrate multiple sources to create protein prioritization ranking."""    def __init__(        self,        results_dir: Path,        explainability_csv: Path,        graph_file: Path,        cohort: str = "ROSMAP",    ):        """        Initialize integrator.        Args:            results_dir: Directory for output files            explainability_csv: Path to protein_attributions.csv            graph_file: Path to protein interaction graph (GraphML)            cohort: Cohort name        """        self.results_dir = Path(results_dir)        self.results_dir.mkdir(parents=True, exist_ok=True)        self.cohort = cohort        # Load data        self.explainability_df = pd.read_csv(explainability_csv)        self.graph = nx.read_graphml(graph_file)        self.proteins = list(self.graph.nodes())        logger.info(f"Loaded {len(self.explainability_df)} proteins from explainability")        logger.info(f"Loaded graph with {len(self.graph.nodes())} nodes and {self.graph.number_of_edges()} edges")    def compute_network_centrality(self) -> Dict[str, float]:        """        Compute multiple network centrality measures.        Returns:            Dict mapping protein ID to centrality score [0, 1]        """        logger.info("Computing network centrality metrics")        centrality_measures = {}        # Degree centrality        degree_cent = nx.degree_centrality(self.graph)        # Betweenness centrality (sample for large graphs)        if len(self.graph) > 5000:            logger.info("Using sampled betweenness centrality (large graph)")            betweenness_cent = nx.betweenness_centrality(self.graph, k=1000)        else:            betweenness_cent = nx.betweenness_centrality(self.graph)        # Closeness centrality (on largest component)        largest_cc = max(nx.connected_components(self.graph), key=len)        subgraph = self.graph.subgraph(largest_cc).copy()        if len(subgraph) > 0:            closeness_cent = nx.closeness_centrality(subgraph)            # Map back to original graph            closeness_cent = {                node: closeness_cent.get(node, 0.0) for node in self.graph.nodes()            }        else:            closeness_cent = {node: 0.0 for node in self.graph.nodes()}        # Combine measures (average)        for protein in self.proteins:            combined = (                degree_cent.get(protein, 0) +                betweenness_cent.get(protein, 0) +                closeness_cent.get(protein, 0)            ) / 3            centrality_measures[protein] = combined        return centrality_measures    def normalize_values(self, values: Dict[str, float]) -> Dict[str, float]:        """        Normalize values to [0, 1] range.        Args:            values: Dict mapping protein to value        Returns:            Dict with normalized values        """        values_list = list(values.values())        if not values_list:            return values        min_val = min(values_list)        max_val = max(values_list)        if max_val == min_val:            return {k: 0.5 for k in values}        normalized = {}        for protein, val in values.items():            normalized[protein] = (val - min_val) / (max_val - min_val)        return normalized    def compute_composite_scores(        self,        gnn_importance: Dict[str, float],        centrality: Dict[str, float],        bootstrap_freq: Dict[str, float],    ) -> Dict[str, float]:        """        Compute composite prioritization scores.        Formula:        score = normalized(GNN) × normalized(centrality) × log(1 + bootstrap_freq)        Args:            gnn_importance: GNN attribution scores            centrality: Network centrality scores            bootstrap_freq: Bootstrap frequency (0-100)        Returns:            Dict mapping protein to composite score        """        logger.info("Computing composite prioritization scores")        # Normalize GNN and centrality        norm_gnn = self.normalize_values(gnn_importance)        norm_centrality = self.normalize_values(centrality)        # Compute stability weight: log(1 + bootstrap_freq)        stability_weights = {            p: np.log(1 + bootstrap_freq.get(p, 0)) / np.log(101)  # Normalize by log(101)            for p in self.proteins        }        # Compute composite score        composite_scores = {}        for protein in self.proteins:            gnn = norm_gnn.get(protein, 0)            cent = norm_centrality.get(protein, 0)            stab = stability_weights.get(protein, 0)            # Multiplicative combination            score = gnn * cent * (1 + stab)  # +1 to ensure non-zero even if stability weight is 0            composite_scores[protein] = score        return composite_scores    def generate_ranking(        self,        bootstrap_freq_file: Optional[Path] = None,    ) -> pd.DataFrame:        """        Generate complete protein ranking.        Args:            bootstrap_freq_file: Optional JSON file with bootstrap frequency        Returns:            DataFrame with proteins ranked by composite score        """        # Extract GNN importance from explainability        gnn_importance = {            row["protein"]: row["mean_attr"]            for _, row in self.explainability_df.iterrows()        }        # Compute centrality        centrality = self.compute_network_centrality()        # Load bootstrap frequency if provided        if bootstrap_freq_file and bootstrap_freq_file.exists():            with open(bootstrap_freq_file, "r") as f:                freq_data = json.load(f)            bootstrap_freq = freq_data.get("bootstrap_frequency_top100", {})            if bootstrap_freq and isinstance(list(bootstrap_freq.values())[0], float):                # Already normalized                bootstrap_freq = {p: f * 100 for p, f in bootstrap_freq.items()}        else:            # Use bootstrap frequency from explainability if available            if "bootstrap_frequency_top100" in self.explainability_df.columns:                bootstrap_freq = {                    row["protein"]: row["bootstrap_frequency_top100"]                    for _, row in self.explainability_df.iterrows()                }            else:                bootstrap_freq = {p: 50 for p in self.proteins}  # Default to 50        # Normalize component scores        norm_gnn = self.normalize_values(gnn_importance)        norm_centrality = self.normalize_values(centrality)        # Compute composite scores        composite_scores = self.compute_composite_scores(            gnn_importance,            centrality,            bootstrap_freq,        )        # Create ranking DataFrame        ranking_data = []        for protein in self.proteins:            ranking_data.append({                "protein": protein,                "composite_score": composite_scores.get(protein, 0),                "gnn_importance": norm_gnn.get(protein, 0),                "centrality": norm_centrality.get(protein, 0),                "stability": bootstrap_freq.get(protein, 0),            })        ranking_df = pd.DataFrame(ranking_data).sort_values("composite_score", ascending=False)        return ranking_df    def map_proteins_to_genes(self, ranking_df: pd.DataFrame) -> pd.DataFrame:        """        Map protein IDs to gene names.        Args:            ranking_df: Ranking DataFrame with protein column        Returns:            DataFrame with gene column added        """        # For now, use protein ID as gene name        # In a real scenario, load from UniProt or annotation file        ranking_df = ranking_df.copy()        # Simple mapping: remove ENSEMBL prefix if present        ranking_df["gene"] = ranking_df["protein"].apply(            lambda x: x.replace("ENSP", "").replace("_HUMAN", "").split(".")[0]            if "ENSP" in x or "_HUMAN" in x            else x        )        return ranking_df[["protein", "gene", "composite_score", "gnn_importance", "centrality", "stability"]]    def get_top_k_subgraph(self, ranking_df: pd.DataFrame, k: int = 50) -> nx.Graph:        """        Extract subgraph of top-k proteins and their interactions.        Args:            ranking_df: Ranking DataFrame            k: Number of top proteins        Returns:            NetworkX subgraph        """        top_proteins = set(ranking_df.head(k)["protein"].values)        # Get all nodes and edges among top proteins        subgraph_nodes = top_proteins.copy()        # Add neighbors of top proteins (for context)        for protein in list(top_proteins):            if protein in self.graph:                neighbors = set(self.graph.neighbors(protein))                # Only add neighbors that have edges within top proteins                for neighbor in neighbors:                    if neighbor in self.graph:                        for edge_neighbor in self.graph.neighbors(neighbor):                            if edge_neighbor in top_proteins:                                subgraph_nodes.add(neighbor)                                break        subgraph = self.graph.subgraph(subgraph_nodes).copy()        logger.info(f"Subgraph has {len(subgraph.nodes())} nodes and {subgraph.number_of_edges()} edges")        return subgraph    def plot_disease_module(        self,        ranking_df: pd.DataFrame,        k: int = 50,        figsize: Tuple[int, int] = (16, 12),    ) -> Figure:        """        Visualize top-k disease module network.        Args:            ranking_df: Ranking DataFrame            k: Number of top proteins            figsize: Figure size        Returns:            Matplotlib figure        """        logger.info(f"Creating network visualization for top-{k} proteins")        # Get subgraph        subgraph = self.get_top_k_subgraph(ranking_df, k=k)        # Map proteins to scores for coloring        top_proteins = set(ranking_df.head(k)["protein"].values)        protein_scores = {            row["protein"]: row["composite_score"]            for _, row in ranking_df.iterrows()            if row["protein"] in top_proteins        }        # Layout        if len(subgraph) > 1:            pos = nx.spring_layout(subgraph, k=0.5, iterations=50, seed=42)        else:            pos = {list(subgraph.nodes())[0]: (0, 0)}        # Create figure        fig, ax = plt.subplots(figsize=figsize)        # Color nodes by score (top-20 in red, others in blue)        top_20 = set(ranking_df.head(20)["protein"].values)        node_colors = [            "red" if node in top_20 else "steelblue"            for node in subgraph.nodes()        ]        # Size nodes by score        node_sizes = [            300 + 2000 * protein_scores.get(node, 0)            for node in subgraph.nodes()        ]        # Draw network        nx.draw_networkx_nodes(            subgraph, pos,            node_color=node_colors,            node_size=node_sizes,            alpha=0.8,            ax=ax,        )        nx.draw_networkx_edges(            subgraph, pos,            alpha=0.3,            edge_color="gray",            ax=ax,        )        # Labels for top-20        top_20_labels = {            node: node.split("_")[0] if "_" in node else node[:8]            for node in subgraph.nodes()            if node in top_20        }        nx.draw_networkx_labels(            subgraph, pos,            labels=top_20_labels,            font_size=8,            font_weight="bold",            ax=ax,        )        ax.set_title(f"AD Disease Module Network (Top {k} Proteins)", fontsize=16, fontweight="bold")        ax.legend(            [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10),             plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue', markersize=10)],            ["Top-20 Proteins", "Supporting Proteins"],            loc="upper left",            fontsize=12,        )        ax.axis("off")        plt.tight_layout()        return fig    def plot_top_20_table(self, ranking_df: pd.DataFrame) -> Figure:        """        Create publication-ready table of top-20 proteins.        Args:            ranking_df: Ranking DataFrame        Returns:            Matplotlib figure with table        """        logger.info("Creating top-20 proteins table")        top_20 = ranking_df.head(20).copy()        # Round scores to 3 decimals        for col in ["composite_score", "gnn_importance", "centrality", "stability"]:            top_20[col] = top_20[col].apply(lambda x: f"{x:.4f}")        # Rename columns for better display        display_df = top_20[[            "protein", "gene", "composite_score", "gnn_importance", "centrality", "stability"        ]].copy()        display_df.columns = [            "Protein ID", "Gene", "Composite Score", "GNN Importance", "Centrality", "Stability"        ]        # Create figure        fig, ax = plt.subplots(figsize=(14, 10))        ax.axis("tight")        ax.axis("off")        # Create table        table = ax.table(            cellText=display_df.values,            colLabels=display_df.columns,            cellLoc="center",            loc="center",            colWidths=[0.15, 0.1, 0.15, 0.15, 0.15, 0.15],        )        table.auto_set_font_size(False)        table.set_fontsize(9)        table.scale(1, 2)        # Style header        for i in range(len(display_df.columns)):            table[(0, i)].set_facecolor("#4472C4")            table[(0, i)].set_text_props(weight="bold", color="white")        # Alternate row colors        for i in range(1, len(display_df) + 1):            for j in range(len(display_df.columns)):                if i % 2 == 0:                    table[(i, j)].set_facecolor("#E7E6E6")                else:                    table[(i, j)].set_facecolor("white")        plt.title(f"Top 20 Prioritized Proteins - {self.cohort}", fontsize=14, fontweight="bold", pad=20)        return fig    def save_results(        self,        ranking_df: pd.DataFrame,        network_fig: Figure,        table_fig: Figure,    ) -> None:        """        Save all results to files.        Args:            ranking_df: Complete ranking DataFrame            network_fig: Network visualization figure            table_fig: Top-20 table figure        """        # Save CSV        csv_path = self.results_dir / f"{self.cohort}_final_protein_targets.csv"        ranking_df.to_csv(csv_path, index=False)        logger.info(f"Saved protein targets to {csv_path}")        # Save network visualization        network_path = self.results_dir / f"{self.cohort}_disease_module_network.png"        network_fig.savefig(network_path, dpi=300, bbox_inches="tight")        logger.info(f"Saved network visualization to {network_path}")        # Save table        table_path = self.results_dir / f"{self.cohort}_top_20_proteins.png"        table_fig.savefig(table_path, dpi=300, bbox_inches="tight")        logger.info(f"Saved top-20 table to {table_path}")        # Save summary JSON        summary = {            "cohort": self.cohort,            "total_proteins_ranked": len(ranking_df),            "top_10_proteins": ranking_df.head(10)[["protein", "gene", "composite_score"]].to_dict(orient="records"),            "top_100_proteins": ranking_df.head(100)[["protein", "gene", "composite_score"]].to_dict(orient="records"),            "score_statistics": {                "mean": float(ranking_df["composite_score"].mean()),                "std": float(ranking_df["composite_score"].std()),                "min": float(ranking_df["composite_score"].min()),                "max": float(ranking_df["composite_score"].max()),            },        }        json_path = self.results_dir / f"{self.cohort}_ranking_summary.json"        with open(json_path, "w") as f:            json.dump(summary, f, indent=2)        logger.info(f"Saved ranking summary to {json_path}")

## run_protein_prioritization

In [ ]:
def run_protein_prioritization(    config_path: str = "config.yaml",    cohorts: Optional[List[str]] = None,) -> Dict:    """    Run protein prioritization ranking for all cohorts.    Args:        config_path: Path to configuration file        cohorts: List of cohort names    Returns:        Dict with ranking results    """    if cohorts is None:        cohorts = ["ROSMAP", "MSBB", "MAYO"]    config = load_config(config_path)    checkpoint_dir = Path(config["logging"]["checkpoint_dir"])    explainability_dir = checkpoint_dir / "explainability"    graphs_dir = Path("data/graphs")    results_dir = checkpoint_dir / "ranking"    results_dir.mkdir(parents=True, exist_ok=True)    logger.info(f"\n{'='*60}")    logger.info("Protein Prioritization Ranking")    logger.info(f"{'='*60}")    all_results = {}    for cohort in cohorts:        logger.info(f"\nProcessing {cohort}...")        try:            # Paths            csv_path = explainability_dir / f"{cohort}_protein_attributions.csv"            graph_path = graphs_dir / f"{cohort}_graph.graphml"            if not csv_path.exists() or not graph_path.exists():                logger.warning(f"Missing files for {cohort}, skipping")                continue            # Run integrator            integrator = ProteinRankingIntegrator(                results_dir=results_dir,                explainability_csv=csv_path,                graph_file=graph_path,                cohort=cohort,            )            # Generate ranking            ranking_df = integrator.generate_ranking()            # Map proteins to genes            ranking_df = integrator.map_proteins_to_genes(ranking_df)            # Create visualizations            network_fig = integrator.plot_disease_module(ranking_df, k=50)            table_fig = integrator.plot_top_20_table(ranking_df)            # Save results            integrator.save_results(ranking_df, network_fig, table_fig)            all_results[cohort] = {                "ranking": ranking_df,                "top_20": ranking_df.head(20),            }            logger.info(f"Ranking complete for {cohort}")            logger.info(f"Top protein: {ranking_df.iloc[0]['gene']} (score={ranking_df.iloc[0]['composite_score']:.4f})")        except Exception as e:            logger.error(f"Error processing {cohort}: {e}")            all_results[cohort] = {"error": str(e)}    logger.info("\nProtein prioritization ranking complete!")    return all_results

## Main Execution

In [ ]:
if __name__ == "__main__":    import argparse    parser = argparse.ArgumentParser(description="Run protein prioritization ranking")    parser.add_argument("--config", default="config.yaml", help="Config file path")    parser.add_argument("--cohorts", nargs="+", default=["ROSMAP", "MSBB", "MAYO"])    args = parser.parse_args()    run_protein_prioritization(config_path=args.config, cohorts=args.cohorts)